# 00 — Data Processing

Run this notebook **first**. Every other notebook in this project loads from
`data/processed/`, so re-run only when raw data, preprocessing, or splits change.

This notebook produces:
- `data/processed/splits.json` — subject-level train/val/test split (seed=42).
- `data/processed/epochs_w0.0-4.0.npz` and `epochs_w0.5-2.5.npz` — epoch tensors.
- `data/processed/data_manifest.json` — full audit trail.

See `PIPELINE_PLAN.md` §4 for the design and rationale.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)


Project root: /Users/luke/Desktop/Personal Projects/eeg-prediction


## 1. Build the deterministic subject split

In [2]:
from src.splits import build_subject_split, save_split, KNOWN_BAD_SUBJECTS

split = build_subject_split()
save_split(split, ROOT / 'data' / 'processed' / 'splits.json')

print('Dropped (known-bad):', split['dropped'])
print(f"Train: {len(split['train'])}  Val: {len(split['val'])}  Test: {len(split['test'])}")
print('First few train subjects:', split['train'][:10])


Dropped (known-bad): [88, 89, 92, 100, 104]
Train: 80  Val: 12  Test: 12
First few train subjects: [1, 3, 4, 5, 6, 7, 8, 9, 12, 13]


## 2. Run preprocessing + epoching for both windows

Implemented in `scripts/run_00_data_processing.py` so the same pipeline is
re-runnable from a terminal. We invoke it here for visibility — re-running
is idempotent.


In [3]:
from scripts.run_00_data_processing import main as run_data_processing
run_data_processing()


Notebook 00 — Data Processing
Raw root      : /Users/luke/Desktop/Personal Projects/eeg-prediction/data/raw
Processed dir : /Users/luke/Desktop/Personal Projects/eeg-prediction/data/processed
Windows       : [(0.0, 4.0), (0.5, 2.5)]
Drop subjects : [88, 89, 92, 100, 104]

Splits → train 80 | val 12 | test 12
Total kept    : 104 subjects (109 - 5 dropped)

Loading + filtering each subject (8–30 Hz, average reference)...
  ... 20 subjects loaded in 3.6s
  ... 40 subjects loaded in 6.4s
  ... 60 subjects loaded in 9.1s
  ... 80 subjects loaded in 11.9s
  ... 100 subjects loaded in 14.6s
Loaded 104 subjects in 15.2s; skipped 0

--- Epoching window w0.0-4.0 (0.0–4.0 s) ---


/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Use

  Total epochs kept: 2485 / 4680 events
  Class balance    : T1 (left) 1257 | T2 (right) 1228
  Balance ratio    : 0.977
  Per-split counts : train 1849 | val 340 | test 296
  Saving → /Users/luke/Desktop/Personal Projects/eeg-prediction/data/processed/epochs_w0.0-4.0.npz

--- Epoching window w0.5-2.5 (0.5–2.5 s) ---


/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Users/luke/Desktop/Personal Projects/eeg-prediction/scripts/run_00_data_processing.py:132: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = ep.get_data(copy=False)  # (n_kept, 64, n_samples)
/Use

  Total epochs kept: 2876 / 4680 events
  Class balance    : T1 (left) 1458 | T2 (right) 1418
  Balance ratio    : 0.973
  Per-split counts : train 2142 | val 373 | test 361
  Saving → /Users/luke/Desktop/Personal Projects/eeg-prediction/data/processed/epochs_w0.5-2.5.npz

Wrote /Users/luke/Desktop/Personal Projects/eeg-prediction/data/processed/data_manifest.json
Done.


## 3. Quick verification: load each window file and inspect shapes

In [4]:
import numpy as np

for win in ['w0.0-4.0', 'w0.5-2.5']:
    arr = np.load(ROOT / 'data' / 'processed' / f'epochs_{win}.npz', allow_pickle=False)
    print(f"{win}: X_3ch={arr['X_3ch'].shape} | X_64ch={arr['X_64ch'].shape} | y={arr['y'].shape} | sfreq={arr['sfreq']}")
    print(f"   class balance: T1={(arr['y']==0).sum()} T2={(arr['y']==1).sum()}")
    print(f"   3-ch order asserted = {list(arr['ch_names_64'][[arr['ch_names_64'].tolist().index(c) for c in ['C3','Cz','C4']]])}")


w0.0-4.0: X_3ch=(2485, 3, 641) | X_64ch=(2485, 64, 641) | y=(2485,) | sfreq=160.0
   class balance: T1=1257 T2=1228
   3-ch order asserted = ['C3', 'Cz', 'C4']
w0.5-2.5: X_3ch=(2876, 3, 321) | X_64ch=(2876, 64, 321) | y=(2876,) | sfreq=160.0
   class balance: T1=1458 T2=1418
   3-ch order asserted = ['C3', 'Cz', 'C4']


## 4. Sanity checks against the plan

- Sampling rate must be 160 Hz.
- No NaN/Inf.
- Class balance within ±15%.
- 3-channel view ordered exactly as `[C3, Cz, C4]`.
- Manifest written.


In [5]:
import json
manifest = json.loads((ROOT / 'data' / 'processed' / 'data_manifest.json').read_text())
print('MNE version :', manifest['mne_version'])
print('Subjects used:', len(manifest['subjects_used']))
for win, info in manifest['windows'].items():
    cb = info['class_counts']
    print(f"  {win}: kept {info['n_kept']}/{info['n_total_events']} epochs ({info['drop_rate']*100:.1f}% drop) "
          f"| T1={cb['T1_left']} T2={cb['T2_right']} | "
          f"train={info['split_counts']['train']} val={info['split_counts']['val']} test={info['split_counts']['test']}")


MNE version : 1.12.1
Subjects used: 104
  w0.0-4.0: kept 2485/4680 epochs (46.9% drop) | T1=1257 T2=1228 | train=1849 val=340 test=296
  w0.5-2.5: kept 2876/4680 epochs (38.5% drop) | T1=1458 T2=1418 | train=2142 val=373 test=361


All artifacts written. Continue with notebooks 01, 02, 03 in any order.